# Generate Training File Lists

This notebook is part of the M2D-CLAP (2025) training scripts.
It generates file lists of training samples by combining audio datasets and excluding evaluation sets (AudioCaps, Clotho, ESC-50, US8K) using WavCaps blacklists.

| Stage | Output file | Datasets |
|---|---|---|
| Stage 1 | `files_A_S_V_S_W_C_X.csv` | AudioSet + VGGSound + WavCaps, excl. AudioCaps/Clotho/ESC-50/US8K |
| Stage 2 | `files_A_S_V_S_W_C_A_C_X_t_E_U.csv` | Stage 1 datasets + AudioCaps train + Clotho train |
| Stage 2.1 (WavCaps only) | `files_w_c_a_c_x_t_e_u.csv` | WavCaps portion of Stage 2 |

## ⚠️ Prerequisites

Before proceeding, please make sure the following (see [data/README.md](../data/README.md)).

- Prepare your file list for AudioSet. -> `../data/files_audioset.csv`
- Prepare your log-mel spectrogram files for AudioSet in `../data/audioset_lms`, VGGSound in `../data/vggsound_lms`, and WavCaps in `../data/wavcaps_lms`.

In [1]:
import warnings; warnings.simplefilter('ignore')
import logging; logging.basicConfig(level=logging.INFO)
import numpy as np
from pathlib import Path
import pandas as pd
import torch
import json

INFO:numexpr.utils:Note: detected 80 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
INFO:numexpr.utils:Note: NumExpr detected 80 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.


## 1. Prepare blacklists using WavCaps

This section also lists all duplicated files from ESC-50/US8K in the Clotho development set.

In [2]:
import requests

# US8K, ESC-50
response = requests.get('https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/blacklist/blacklist_exclude_ub8k_esc50_vggsound.json')
response.raise_for_status()
data = response.json()
# bAS = data['AudioSet']  # We do not use the VGGSound list from  the AudioSet list.
bVGGS = data['AudioSet']
bESCUS = data['FreeSound']
print('VGGSound files in FreeSound:', len(bVGGS), len(set(bVGGS)))
print('ESC-50 & US8K files in FreeSound:', len(bESCUS), len(set(bESCUS)))

# AudioCaps & Clotho
response = requests.get('https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/blacklist/blacklist_exclude_all_ac.json')
response.raise_for_status()
data = response.json()
bAc = data['AudioSet']
bCl = data['FreeSound']
print('AudioCaps files in AudioSet:', len(bAc),  len(set(bAc)))
print('Clotho files in FreeSound:', len(bCl),  len(set(bCl)))

response = requests.get('https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/blacklist/blacklist_exclude_test_ac.json')
response.raise_for_status()
data = response.json()
bAcT = data['AudioSet']  # T = Test
bClT = data['FreeSound']
print('AudioCaps test files in AudioSet:', len(bAcT),  len(set(bAcT)))
print('Clotho test files in FreeSound:', len(bClT),  len(set(bClT)))

# Clotho files
cldev = pd.read_csv('https://zenodo.org/records/4783391/files/clotho_metadata_development.csv', encoding="ISO-8859-1")[['file_name', 'sound_id']]
cldev_ESCUSdup = cldev[cldev.sound_id.isin(bESCUS)]
print('Duplicated ESC50/US8K files in Clotho development:', len(cldev_ESCUSdup))

# MusicCaps files
muc = pd.read_csv('https://huggingface.co/datasets/google/MusicCaps/resolve/main/musiccaps-public.csv')
bMuCaps = muc.ytid.map(lambda x: x+'.wav').to_list()
print('MusicCaps files in AudioSet:', len(bMuCaps))

VGGSound files in FreeSound: 15510 15510
ESC-50 & US8K files in FreeSound: 2219 2219
AudioCaps files in AudioSet: 50725 50725
Clotho files in FreeSound: 5929 5905
AudioCaps test files in AudioSet: 1451 1451
Clotho test files in FreeSound: 2090 2078
Duplicated ESC50/US8K files in Clotho development: 97
MusicCaps files in AudioSet: 5521


### AudioSet available file list

In [3]:
asdf = pd.read_csv('../data/files_audioset.csv')  # Create this file before run here.
asdf['key'] = asdf.file_name.apply(lambda x: Path(x).stem[:11])
len(asdf)

2005132

### VGGSound (VS) available file list

In [4]:
files = [f.relative_to('../data') for f in Path('../data/vggsound_lms/').glob('*.npy')]
vsdf = pd.DataFrame({'file_name': files})
vsdf['key'] = vsdf.file_name.apply(lambda x: Path(x).stem[:11])
len(vsdf), vsdf[:5]

(197956,
                              file_name          key
 0  vggsound_lms/Cmh023qRXcM_000100.npy  Cmh023qRXcM
 1  vggsound_lms/k5qy4F4j8A8_000013.npy  k5qy4F4j8A8
 2  vggsound_lms/_zMzODntmas_000040.npy  _zMzODntmas
 3  vggsound_lms/XD5dPWHYg3U_000010.npy  XD5dPWHYg3U
 4  vggsound_lms/DBkpo9eesb0_000019.npy  DBkpo9eesb0)

In [ ]:
import requests
from pathlib import Path

wavcaps_json_files = [
    'https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/AudioSet_SL/as_final.json',
    'https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/BBC_Sound_Effects/bbc_final.json',
    'https://github.com/XinhaoMei/WavCaps/raw/refs/heads/master/data/json_files/FreeSound/fsd_final.json',
    'https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/SoundBible/sb_final.json'
]

folders, ids = [], []
for url in wavcaps_json_files:
    print(f"Downloading: {url}")
    response = requests.get(url)
    js = response.json()['data']
    folder = Path(url).parent.name
    folders.extend([folder for d in js])
    ids.extend([Path(d['id']).stem for d in js])

wcdf = pd.DataFrame({'file_name': ['wavcaps_lms/'+f+'_flac/'+i+'.npy' for f, i in zip(folders, ids)]})
wcdf['key'] = wcdf.file_name
len(wcdf)

403050

## Remove tests and add Ac/Cl_train

In [7]:
## asxact = AS - AC_test; vsxact = VS - AC_test
# Convert all list as a complete path name

bl_as  = [f'{_id[1:12]}' for _id in bAcT]
print(f'Removing AudioCaps files from the local AudioSet file list (at most {len(bl_as)}).')
asxact = asdf[~asdf.key.isin(bl_as)]
print(f'The original local AudioSet file list ({len(asdf)}) shrank to {len(asxact)}).')

print(f'Removing AudioCaps files from the local VGGSound file list (at most {len(bl_as)}).')
vsxact = vsdf[~vsdf.key.isin(bl_as)]
print(f'The original local VGGSound file list ({len(vsdf)}) shrank to {len(vsxact)}).')

Removing AudioCaps files from the local AudioSet file list (at most 1451).
The original local AudioSet file list (2005132) shrank to 2003689).
Removing AudioCaps files from the local VGGSound file list (at most 1451).
The original local VGGSound file list (197956) shrank to 197752).


In [8]:
## wcxacteu = WavCaps - AC_test - Clotho_test - ESC/US

blacklist_WCXAC_AcT = [f'wavcaps_lms/AudioSet_SL_flac/{_id[:12]}.npy' for _id in bAcT]
blacklist_WCXAC_ClT = [f'wavcaps_lms/FreeSound_flac/{_id}.npy' for _id in bClT]
blacklist_WCXAC_EU = [f'wavcaps_lms/FreeSound_flac/{_id}.npy' for _id in bESCUS]
print(len(blacklist_WCXAC_AcT), len(blacklist_WCXAC_ClT), len(blacklist_WCXAC_EU))

print('Original WavCaps:', len(wcdf))
wcxa = wcdf[~wcdf.key.isin(blacklist_WCXAC_AcT)].copy()
print('wcxa = Excluded all AudioCaps files:', len(wcxa))
wcxac = wcxa[~wcxa.key.isin(blacklist_WCXAC_ClT)]
print('wcxac = Excluded all AudioCaps and Clotho files:', len(wcxac))
wcxacteu = wcxac[~wcxac.key.isin(blacklist_WCXAC_EU)]
print('wcxacteu = Excluded all ESC/US files:', len(wcxacteu))

1451 2090 2219
Original WavCaps: 403050
wcxa = Excluded all AudioCaps files: 402789
wcxac = Excluded all AudioCaps and Clotho files: 400956
wcxacteu = Excluded all ESC/US files: 398799


In [9]:
### acxeu = Ac_train + Cl_train - ESC - US

ac_train = pd.read_csv('https://raw.githubusercontent.com/jaeyeonkim99/EnCLAP/refs/heads/main/csv/audiocaps/train.csv')[['file_path']]
ac_train.file_path = ac_train.file_path.apply(lambda x: x.replace('train/', 'audiocaps_lms/train/'))
ac_train = ac_train.sort_values('file_path')
cl_train = pd.read_csv('https://raw.githubusercontent.com/jaeyeonkim99/EnCLAP/refs/heads/main/csv/clotho/train.csv')[['file_path']]
cl_train = cl_train.drop_duplicates()  # 5 rows duplicate for each sample
cl_train.file_path = cl_train.file_path.apply(lambda x: x.replace('train/', 'clotho_lms/development/'))
print('AudioCaps training samples:', len(ac_train), 'Clotho training samples:', len(cl_train))
## Clotho data correction
cl_train.file_path =  cl_train.file_path.apply(lambda x: x.replace('clotho_lms/development/20061208.npyes.02.npy', 'clotho_lms/development/20061208.waves.02.npy'))

ac_cl_train = pd.concat([ac_train, cl_train], ignore_index=True)
ac_cl_train.columns = ['file_name']
print('All AudioCaps & Clotho training samples:', len(ac_cl_train))

# Remove ESC/US
blacklist_ClothoESCUS = [f'clotho_lms/development/{f[:-4]}.npy' for f in cldev_ESCUSdup.file_name.values]
acxeu = ac_cl_train[~ac_cl_train.file_name.isin(blacklist_ClothoESCUS)]
print('All AudioCaps & Clotho training samples - ESC/US files:', len(acxeu))

AudioCaps training samples: 49274 Clotho training samples: 3835
All AudioCaps & Clotho training samples: 53109
All AudioCaps & Clotho training samples - ESC/US files: 53012


### asxac = AS - AC

In [10]:
# Convert all list as a complete path name

bl_as  = [f'{_id[1:12]}' for _id in bAc]
print(f'Removing AudioCaps files from the local AudioSet file list (at most {len(bl_as)}).')
asxac = asdf[~asdf.key.isin(bl_as)]
print(f'The original local AudioSet file list ({len(asdf)}) shrank to {len(asxac)}).')

print(f'Removing AudioCaps files from the local VGGSound file list (at most {len(bl_as)}).')
vsxac = vsdf[~vsdf.key.isin(bl_as)]
print(f'The original local VGGSound file list ({len(vsdf)}) shrank to {len(vsxac)}).')

Removing AudioCaps files from the local AudioSet file list (at most 50725).
The original local AudioSet file list (2005132) shrank to 1954821).
Removing AudioCaps files from the local VGGSound file list (at most 50725).
The original local VGGSound file list (197956) shrank to 190663).


### wcxaceu = WavCaps - AC all - Clotho all - ESC/US

In [11]:
blacklist_WCXAC_Ac = [f'wavcaps_lms/AudioSet_SL_flac/{_id[:12]}.npy' for _id in bAc]
blacklist_WCXAC_Cl = [f'wavcaps_lms/FreeSound_flac/{_id}.npy' for _id in bCl]
blacklist_WCXAC_EU = [f'wavcaps_lms/FreeSound_flac/{_id}.npy' for _id in bESCUS]
print(len(blacklist_WCXAC_Ac), len(blacklist_WCXAC_Cl), len(blacklist_WCXAC_EU))

print('Original WavCaps:', len(wcdf))
wcxa = wcdf[~wcdf.key.isin(blacklist_WCXAC_Ac)].copy()
print('wcxa = Excluded all AudioCaps files:', len(wcxa))
wcxac = wcxa[~wcxa.key.isin(blacklist_WCXAC_Cl)]
print('wcxac = Excluded all AudioCaps and Clotho files:', len(wcxac))
wcxaceu = wcxac[~wcxac.key.isin(blacklist_WCXAC_EU)]
print('wcxaceu = Excluded all ESC/US files:', len(wcxaceu))

50725 5929 2219
Original WavCaps: 403050
wcxa = Excluded all AudioCaps files: 394051
wcxac = Excluded all AudioCaps and Clotho files: 388791
wcxaceu = Excluded all ESC/US files: 386731


### acxeu = Ac_train + Cl_train - ESC - US

In [12]:
ac_train = pd.read_csv('https://raw.githubusercontent.com/jaeyeonkim99/EnCLAP/refs/heads/main/csv/audiocaps/train.csv')[['file_path']]
ac_train.file_path = ac_train.file_path.apply(lambda x: x.replace('train/', 'audiocaps_lms/train/'))
ac_train = ac_train.sort_values('file_path')
cl_train = pd.read_csv('https://raw.githubusercontent.com/jaeyeonkim99/EnCLAP/refs/heads/main/csv/clotho/train.csv')[['file_path']]
cl_train = cl_train.drop_duplicates()  # 5 rows duplicate for each sample
cl_train.file_path = cl_train.file_path.apply(lambda x: x.replace('train/', 'clotho_lms/development/'))
print('AudioCaps training samples:', len(ac_train), 'Clotho training samples:', len(cl_train))
## Clotho data correction
cl_train.file_path =  cl_train.file_path.apply(lambda x: x.replace('clotho_lms/development/20061208.npyes.02.npy', 'clotho_lms/development/20061208.waves.02.npy'))

ac_cl_train = pd.concat([ac_train, cl_train], ignore_index=True)
ac_cl_train.columns = ['file_name']
print('All AudioCaps & Clotho training samples:', len(ac_cl_train))

# Remove ESC/US
blacklist_ClothoESCUS = [f'clotho_lms/development/{f[:-4]}.npy' for f in cldev_ESCUSdup.file_name.values]
acxeu = ac_cl_train[~ac_cl_train.file_name.isin(blacklist_ClothoESCUS)]
print('All AudioCaps & Clotho training samples - ESC/US files:', len(acxeu))

AudioCaps training samples: 49274 Clotho training samples: 3835
All AudioCaps & Clotho training samples: 53109
All AudioCaps & Clotho training samples - ESC/US files: 53012


### wcacxteu = WavCaps - AC - Cl + AC_train + Cl_train - ESC50 - US8K

In [13]:
wcacxteu = pd.concat([wcxaceu, acxeu], ignore_index=True)
print('WavCaps - all (AudioCaps&Clotho samples) + AudioCaps & Clotho training samples:', len(wcacxteu))
wcacxteu['file_name'].to_csv('../data/files_w_c_a_c_x_t_e_u.csv', index=None)

WavCaps - all (AudioCaps&Clotho samples) + AudioCaps & Clotho training samples: 439743


### A_S_V_S_W_C_A_C_X_t_E_U = AudioSet - AC + VGGSound - AC + wcacxteu

In [14]:
aswcacxteu = pd.concat([asxac, vsxac, wcacxteu], ignore_index=True)
print('AudioSet + VGGSound + WavCaps - all (AudioCaps&Clotho samples) + AudioCaps & Cloth otraining samples:', len(aswcacxteu))
aswcacxteu['file_name'].to_csv('../data/files_A_S_V_S_W_C_A_C_X_t_E_U.csv', index=None)

AudioSet + VGGSound + WavCaps - all (AudioCaps&Clotho samples) + AudioCaps & Cloth otraining samples: 2585227


### A_S_V_S_W_C_X = AudioSet - AC + VGGSound - AC + WavCaps - AC - Cl - ESC/US

In [15]:
asvswcx = pd.concat([asxac, vsxac, wcxaceu], ignore_index=True)
print('AudioSet + VGGSound + WavCaps - all AudioCaps/Clotho/ESC-50/US8K samples:', len(asvswcx))
asvswcx['file_name'].to_csv('../data/files_A_S_V_S_W_C_X.csv', index=None)

AudioSet + VGGSound + WavCaps - all AudioCaps/Clotho/ESC-50/US8K samples: 2532215
